[![../img/pythonista.png](../img/pythonista.png)](https://www.pythonista.io)

# Laboratorio: RAG como herramienta.

**py501 · Aplicaciones Agénticas con LangChain** · Pythonista® · Instructor: José Luis Chiquete · Módulo 3: *Herramientas, contexto y RAG*

> 🔑 **Versión con soluciones.** Solo para el instructor o para revisión posterior.

En este *notebook* se reemplazará la lectura de políticas completas por **búsqueda semántica**: fragmentar, indexar con *embeddings*, exponer la búsqueda como herramienta y medir el efecto del reordenamiento (*reranking*).

## 🧭 Cómo usar este *notebook*.

* Ejecuta las celdas **en orden** (`Shift + Enter`).
* **Ejercicio básico:** lo que todos deben dominar.
* **Reto (opcional):** para quien quiere profundizar. Se puede resolver como práctica.
* Las celdas `# TU CÓDIGO AQUÍ` y *✍️ Tu respuesta* son para ti.
* Cada ejecución llama a *Gemini* (el modelo del curso): las respuestas varían entre ejecuciones y consumen la cuota de tu `GOOGLE_API_KEY`.

### Preparación del entorno.

* La siguiente celda localizará `data/`, cargará `.env` si existe (solo en ejecución local), definirá el modelo del curso (*Gemini*) y verificará que exista la clave `GOOGLE_API_KEY`. Si falla, sigue `TUTORIAL_SECRETOS.md`.

In [ ]:
import os

from dotenv import load_dotenv

RAIZ = next(p for p in (os.curdir, os.pardir)
            if os.path.isdir(os.path.join(p, "data", "politicas")))
DATA = os.path.join(RAIZ, "data")
POLITICAS = os.path.join(DATA, "politicas")
load_dotenv(os.path.join(RAIZ, ".env"))   # solo tiene efecto en ejecución local; en Codespaces se usan los secretos

# Este curso usa Gemini. Cualquier proveedor funciona cambiando la cadena "proveedor:modelo" (ver notebook 03).
MODELO = os.environ.get("MODELO", "google_genai:gemini-3.5-flash")
EMBEDDINGS = os.environ.get("EMBEDDINGS", "google_genai:gemini-embedding-001")

if MODELO.startswith("google_genai") and not (os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY")):
    raise RuntimeError("Falta GOOGLE_API_KEY. Sigue TUTORIAL_SECRETOS.md: guárdala como secreto de Codespaces "
                       "con acceso a este repositorio y vuelve a abrir el Codespace.")
MODELO

* La siguiente celda importará las bibliotecas de este *notebook*.

In [ ]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.embeddings import init_embeddings
from langchain.tools import tool
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
from pydantic import BaseModel

---
## 1. Fragmentación.

RAG (*Retrieval-Augmented Generation*) recupera fragmentos relevantes y los entrega al modelo como contexto. Su calidad depende sobre todo de **cómo se fragmenta**: fragmentos muy chicos pierden contexto; muy grandes diluyen la información relevante.

Aquí se fragmenta en dos pasos: primero por encabezado de *Markdown* (para conservar la sección como metadato) y después por tamaño.

* La siguiente celda fragmentará las tres políticas y mostrará el primer fragmento con sus metadatos.

In [ ]:
def fragmentar(chunk_size: int = 600, chunk_overlap: int = 80) -> list:
    por_encabezado = MarkdownHeaderTextSplitter(headers_to_split_on=[("#", "politica"), ("##", "seccion")])
    por_tamano = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    fragmentos = []
    for archivo in sorted(os.listdir(POLITICAS)):
        with open(os.path.join(POLITICAS, archivo), encoding="utf-8") as f:
            docs = por_encabezado.split_text(f.read())
        for doc in docs:
            doc.metadata["archivo"] = archivo
        fragmentos.extend(por_tamano.split_documents(docs))
    return fragmentos


fragmentos = fragmentar()
len(fragmentos), fragmentos[0]

---
## 2. Índice vectorial.

* La siguiente celda calculará los *embeddings* de cada fragmento, los guardará en un índice en memoria y hará una búsqueda de prueba.

In [ ]:
indice = InMemoryVectorStore(init_embeddings(EMBEDDINGS))
indice.add_documents(fragmentos)

for doc, puntaje in indice.similarity_search_with_score("¿cuánto puedo gastar en hotel?", k=3):
    print(f"{puntaje:.3f} | {doc.metadata['archivo']} | {doc.metadata.get('seccion')}")

**Observa:** el puntaje es la similitud coseno entre la pregunta y cada fragmento. La pregunta no contiene la palabra "hospedaje" y aun así recupera esa sección: la búsqueda es por **significado**, no por palabras.

---
## 3. La búsqueda como herramienta.

* La siguiente celda creará la herramienta `buscar_politicas` y un agente que la usa.

In [ ]:
@tool
def buscar_politicas(consulta: str) -> str:
    """Busca en las políticas internas los fragmentos relevantes para una pregunta.

    Úsala antes de responder cualquier duda sobre vacaciones, equipo de cómputo,
    gastos de viaje u otras reglas internas.
    """
    docs = indice.similarity_search(consulta, k=4)
    return "\n\n---\n\n".join(f"[{d.metadata['archivo']} | {d.metadata.get('seccion', '')}]\n{d.page_content}" for d in docs)


agente = create_agent(MODELO, tools=[buscar_politicas], system_prompt=(
    "Eres el asistente de soporte interno de Andina Logística. "
    "Busca en las políticas antes de responder y cita el archivo de origen."))
r = agente.invoke({"messages": [{"role": "user", "content":
    "Viajo a Monterrey 3 noches, ¿cuál es mi tope de hotel y en cuánto tiempo compruebo?"}]})
r["messages"][-1].text

**RAG clásico vs. RAG agéntico.** En el clásico, el código siempre busca con la pregunta del usuario y pega los resultados en el *prompt*. En el agéntico, el **agente decide** si buscar, qué buscar y si buscar otra vez. El agéntico maneja mejor preguntas compuestas (como la anterior, que toca dos secciones) a cambio de más llamadas.

**💡 Ideas clave de la sección 3:**
* Fragmentar por estructura (encabezados) y luego por tamaño conserva el contexto.
* La herramienta devuelve el origen de cada fragmento para que el agente pueda citarlo.
* En RAG agéntico, el agente puede reformular y repetir la búsqueda.

---
## 4. Reordenamiento.

La búsqueda vectorial recupera por **similitud**, no por **utilidad**. Un reordenador recupera más candidatos (por ejemplo, 8) y se queda con los mejores. Aquí se usa el propio modelo como reordenador.

* La siguiente celda definirá un reordenador basado en el modelo y comparará los resultados con y sin reordenar.

In [ ]:
class Puntajes(BaseModel):
    puntajes: list[int]


juez = init_chat_model(MODELO).with_structured_output(Puntajes)


def reordenar(consulta: str, docs: list, top: int = 3) -> list:
    listado = "\n\n".join(f"[{i}] {d.page_content}" for i, d in enumerate(docs))
    resultado = juez.invoke(
        f"Pregunta: {consulta}\n\nCalifica de 0 a 10 qué tan útil es cada fragmento para responderla. "
        f"Devuelve una lista con {len(docs)} enteros en el mismo orden.\n\n{listado}")
    orden = sorted(zip(resultado.puntajes, docs), key=lambda par: par[0], reverse=True)
    return [d for _, d in orden[:top]]


consulta = "Perdí mi laptop, ¿qué hago y en cuánto tiempo?"
sin = indice.similarity_search(consulta, k=3)
con = reordenar(consulta, indice.similarity_search(consulta, k=8))
{"sin": [d.metadata.get("seccion") for d in sin], "con": [d.metadata.get("seccion") for d in con]}

### Ejercicio 1 · Herramienta con reordenamiento.

Crea `buscar_politicas_reordenada` que recupere 8 fragmentos, los reordene con `reordenar` y devuelva los 3 mejores. Compara las respuestas de un agente con cada herramienta en las 4 preguntas de `PREGUNTAS`.

**🎯 Objetivo:** integrar el reordenamiento en una herramienta. **Qué se espera aprender:** a decidir si la mejora justifica la llamada extra por búsqueda.

* La siguiente celda muestra una solución posible.

In [ ]:
PREGUNTAS = [
    "Perdí mi laptop, ¿qué hago y en cuánto tiempo?",
    "¿Se reembolsa un taxi sin factura?",
    "¿Puedo instalar un programa que no está aprobado?",
    "¿Cuántos días de vacaciones tengo con 7 años de antigüedad?",
]


@tool
def buscar_politicas_reordenada(consulta: str) -> str:
    """Busca en las políticas internas los fragmentos más útiles para una pregunta."""
    docs = reordenar(consulta, indice.similarity_search(consulta, k=8))
    return "\n\n---\n\n".join(f"[{d.metadata['archivo']}]\n{d.page_content}" for d in docs)


agente_reordenado = create_agent(MODELO, tools=[buscar_politicas_reordenada],
                                 system_prompt="Busca en las políticas antes de responder y cita el archivo.")
for p in PREGUNTAS:
    a = agente.invoke({"messages": [{"role": "user", "content": p}]})["messages"][-1].text
    b = agente_reordenado.invoke({"messages": [{"role": "user", "content": p}]})["messages"][-1].text
    print(f"### {p}\nSIN: {a}\nCON: {b}\n")

### Ejercicio 2 · Tamaño de fragmento.

Reconstruye el índice con `fragmentar(chunk_size=200)` y con `fragmentar(chunk_size=1200)`. Para cada uno, busca `"¿Cuántos días de vacaciones tengo con 7 años de antigüedad?"` con `k=2`. ¿Qué tamaño recupera la información completa?

**🎯 Objetivo:** observar el efecto del tamaño de fragmento. **Qué se espera aprender:** que no hay un tamaño universal; se elige midiendo con preguntas reales.

* La siguiente celda muestra una solución posible.

In [ ]:
for tamano in (200, 1200):
    idx = InMemoryVectorStore(init_embeddings(EMBEDDINGS))
    idx.add_documents(fragmentar(chunk_size=tamano, chunk_overlap=min(80, tamano // 4)))
    print(f"--- chunk_size={tamano}")
    for d in idx.similarity_search("¿Cuántos días de vacaciones tengo con 7 años de antigüedad?", k=2):
        print(repr(d.page_content[:200]))

**✅ Respuesta sugerida:**

Con 200 caracteres, la regla de antigüedad queda partida: un fragmento tiene "del segundo al quinto año" y otro "a partir del sexto", y con `k=2` puede faltar uno. Con 1200, la sección completa de "Días por antigüedad" cabe en un fragmento. Con documentos más largos, 1200 diluiría la información; por eso se mide.

### Reto 1 · Costo del reordenamiento.

En *LangSmith*, compara una ejecución de `agente` y una de `agente_reordenado` con la misma pregunta: tokens totales y latencia. ¿Cuánto cuesta cada búsqueda reordenada?

**🎯 Objetivo:** cuantificar el costo de una mejora. **Qué se espera aprender:** a leer trazas para tomar decisiones de arquitectura.

**✅ Respuesta sugerida:**

El reordenamiento agrega una llamada al modelo por búsqueda con los 8 fragmentos completos: típicamente varios cientos a un par de miles de tokens de entrada extra y entre 0.5 y 2 segundos más, según el modelo. Conviene cuando las respuestas sin reordenar fallan; si no, es costo sin beneficio.

---
## ✅ Cierre.

**Práctica:** terminar el ejercicio 1.

**Siguiente *notebook*:** `07_mcp.ipynb`.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>